# Session 3 — PyTorch tensors and autograd

Companion to [../numpy_pytorch_schedule.md](../numpy_pytorch_schedule.md). A tensor is a NumPy array that (a) runs on GPU and (b) records ops so gradients can be computed. Nearly every NumPy op has an identically-named torch op, so Sessions 1–2 transfer directly. This session covers only what's **new**.

In [1]:
import torch
import torch.nn.functional as F

x = torch.arange(12).reshape(3, 4)
print(x.shape, x.dtype)
print("dim/keepdim (not axis/keepdims):", x.sum(dim=1, keepdim=True).shape)

torch.Size([3, 4]) torch.int64
dim/keepdim (not axis/keepdims): torch.Size([3, 1])


Naming diffs from NumPy: `dim` not `axis`, `keepdim` not `keepdims`. Broadcasting, `@`, and `einsum` are identical. NumPy interop shares memory:

In [ ]:
import numpy as np
t = torch.from_numpy(np.ones(3))     # array -> tensor
a = t.numpy()                         # tensor -> array
print(type(t), type(a))

## dtype and device

Two attributes NumPy lacks, both critical. Ops require all tensors on the **same** device ("expected all tensors on the same device" is the error you'll see most).

In [ ]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
x = torch.randn(2, 3, dtype=torch.float32, device=device)
print(x.dtype, x.device)

## `view` vs `reshape` vs `.contiguous()`

- **`view`** — a new shape over the **same** memory; **never copies**. Works only if the layout allows it (in practice: contiguous), else it **errors**. So it *guarantees* no-copy + shared storage — and fails loudly if your layout assumption is wrong.
- **`reshape`** — **tries** `view`; if the layout won't allow it, it silently makes a **contiguous copy** and views that. Never errors, but you don't know whether you got a view (shared, free) or a copy (`O(n)`). Mental model: `reshape ≈ view() if possible, else contiguous().view()`.
- After `transpose`/`permute`, memory isn't contiguous, so `view` errors — call `.contiguous()` first (or use `reshape`). You'll hit this exact error in multi-head attention (Session 5).

**When to call `.contiguous()` — mostly correctness, rarely speed.**
- **Correctness (common):** an op *requires* contiguous input (`view` is the main one). Use `.contiguous()` (or `reshape`, which copies for you).
- **Performance (rare, profile-driven):** worth a copy only when a **memory-bandwidth-bound** op has an **ugly stride** *and* you **reuse** the tensor across many ops (one copy amortizes) — or a profile flags a specific strided op as hot. Compute-bound, stride-tolerant kernels (matmul, `scaled_dot_product_attention`) handle strides fine — don't copy for them. **Never add `.contiguous()` speculatively:** it's an `O(n)` copy + extra memory that can *slow* an otherwise-fine pipeline.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print("view ok:", x.view(2, 6).shape)
xt = x.transpose(0, 1)               # (4,3), NOT contiguous
try:
    xt.view(12)
except RuntimeError as e:
    print("view error:", str(e)[:60], "...")
print("contiguous().view():", xt.contiguous().view(12).shape)
print("reshape (copies):", xt.reshape(12).shape)

## Autograd — the whole point

Set `requires_grad=True` on leaves, do a computation, call `.backward()` on a **scalar**, read `.grad`. This is the mechanism behind Part 2.2.

**Why a scalar?** `backward()` starts the chain rule from the seed `d(out)/d(out) = 1`, so it needs a *single* number to begin from. Call it on a non-scalar and you get *"grad can be implicitly created only for scalar outputs"* (you'd have to pass an explicit `gradient` seed). A **loss is a scalar tensor** — a 0-dim `Tensor` (shape `()`, one element) that still carries `requires_grad` and a `grad_fn`; it's a *tensor*, not a Python float (`.item()` converts it to a float). It comes out scalar because losses reduce to one number (e.g. `F.cross_entropy`'s default `reduction='mean'` averages over all examples), which is exactly what lets `loss.backward()` be called with no arguments.

In [2]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()                   # y = 14; dy/dx_i = 2 x_i
y.backward()
print("x.grad:", x.grad)             # [2., 4., 6.] == 2*x

x.grad: tensor([2., 4., 6.])


**`.grad` accumulates** (`+=`) across `backward()` calls — it is *not* overwritten. This is why training loops call `zero_grad()` every step. Watch it double:

In [3]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
(x ** 2).sum().backward(); print("after 1st:", x.grad)
(x ** 2).sum().backward(); print("after 2nd:", x.grad)   # doubled!
x.grad = None                        # (or optimizer.zero_grad())
(x ** 2).sum().backward(); print("after zero+backward:", x.grad)

after 1st: tensor([2., 4., 6.])
after 2nd: tensor([ 4.,  8., 12.])
after zero+backward: tensor([2., 4., 6.])


**`torch.no_grad()`** disables graph-tracking for inference (faster, no backward memory); **`.detach()`** cuts a tensor out of the graph.

In [4]:
w = torch.randn(3, requires_grad=True)
with torch.no_grad():
    y = (w * 2).sum()
print("tracked under no_grad?", y.requires_grad)   # False
print("detach requires_grad?", (w.detach()).requires_grad)  # False

tracked under no_grad? False
detach requires_grad? False


## Self-check

1. `transpose` then `.view(...)` errors ("not contiguous"). Two fixes?
2. Call `backward()` twice without zeroing — what happens to `.grad`, and what prevents it?
3. When wrap in `torch.no_grad()`, and what does it save?

**Answers.** (1) `.contiguous().view(...)` or just `.reshape(...)`. (2) Gradients **accumulate** (double); `optimizer.zero_grad()` (or `x.grad=None`) each step prevents it — accumulation is deliberate (micro-batching) but must reset per real step. (3) Wrap any forward you won't `backward()` (inference/eval); it skips building the autograd graph — saves memory and time.

## Exercise — differentiable attention

Port Session 2's attention to tensors with `requires_grad=True`, reduce to a scalar, `backward()`, and confirm `Q.grad` has `Q`'s shape.

In [ ]:
S, d = 4, 8
Q = torch.randn(S, d, requires_grad=True)
K = torch.randn(S, d, requires_grad=True)
V = torch.randn(S, d, requires_grad=True)

A = F.softmax((Q @ K.T) / (d ** 0.5), dim=-1)    # stable softmax built-in
out = A @ V
out.sum().backward()
print("Q.grad shape:", Q.grad.shape)             # torch.Size([4, 8])
print("non-None:", Q.grad is not None)

`F.softmax(..., dim=-1)` does the max-subtraction internally. The Part-2 rule holds: `Q.grad` has exactly `Q`'s shape. Session 4 wraps this into a trainable `nn.Module`.